# Play DQN on Atari Assault
This notebook loads a trained DQN model and runs a visual demo episode.

What this notebook does:
- Rebuild the same DQN architecture used in training.
- Load a saved checkpoint from the DQN model folder.
- Interact with the environment using mostly greedy actions.
- Render frames with OpenCV and print reward statistics in real time.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ale_py.vector_env import AtariVectorEnv
import cv2
import numpy as np
import os

In [ ]:
class DeepQNetwork(nn.Module):
    def __init__(self, num_stack, num_action):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=num_stack, out_channels=32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1)
        self.lin1 = nn.Linear(3136, 512) 
        self.lin2 = nn.Linear(512, num_action)
    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.relu(self.conv2(X))
        X = F.relu(self.conv3(X))
        X = torch.flatten(X, start_dim=1)  # Flatten convolution output
        X = F.relu(self.lin1(X))
        return self.lin2(X)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
net = DeepQNetwork(num_stack=4, num_action=7)
net = net.to(device)
net.load_state_dict(torch.load(os.path.join("..", "models", "dqn", "timesteps_10000000.pt"), map_location=device))

<All keys matched successfully>

In [18]:
envs = AtariVectorEnv(
    game="assault",
    num_envs=1,
    stack_num=4
)

In [19]:
def run_env(envs):
    observations, _ = envs.reset()
    total_rewards = 0.0
    step = 0

    while True:
        if np.random.rand() > 0.05:
            with torch.no_grad():
                observations = torch.tensor(observations, dtype=torch.float32)/ 255.0
                observations = observations.to(device)
                actions = net(observations).argmax(dim=1)
                actions = np.array(actions.cpu())
        else:
            actions = envs.action_space.sample()
        observations, rewards, terminations, truncations, infos = envs.step(actions)

        step += 1
        total_rewards += float(rewards[0])

        frame = observations[0,0,:,:].reshape(84, 84)
        frame_big = cv2.resize(
            frame,
            None,
            fx=4,
            fy=4,
            interpolation=cv2.INTER_NEAREST
        )
        cv2.imshow("obs", frame_big)

        print(
            f"step={step} | reward={float(rewards[0]):.2f} | total_reward={total_rewards:.2f}",
            end="\r",
            flush=True,
        )

        key = cv2.waitKey(100) & 0xFF
        if key == ord("q") or terminations[0] or truncations[0]:
            break

    print(f"\nEpisode done | steps={step} | total_reward={total_rewards:.2f}")
    cv2.destroyAllWindows()

In [22]:
run_env(envs)

C:\Users\15221\AppData\Local\Temp\ipykernel_30644\3855334176.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  actions = np.array(actions.cpu())


step=1599 | reward=0.00 | total_reward=89.00
Episode done | steps=1599 | total_reward=89.00
